In [25]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from dotenv import load_dotenv

In [26]:
load_dotenv()

True

### Tool Creation

In [27]:
@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b, this tool returns their product"""
    return a*b

In [28]:
print(multiply.invoke({'a':2, 'b':3}))

6


In [29]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given 2 numbers a and b, this tool returns their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


### Tool Binding

In [30]:
llm = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite')

In [31]:
llm_with_tools = llm.bind_tools([multiply])

In [32]:
result = llm_with_tools.invoke('hi how are you')

display(result.text)  # tool will not be used here
display(result.tool_calls)  # no output since LLM will decide that tool does not need to be called

"I'm doing well, thank you for asking! How are you doing today? Is there anything I can help you with?"

[]

In [33]:
result = llm_with_tools.invoke('can you multiply 3 with 10')

display(result.text)    # no output
display(result.tool_calls[0])  # LLM will suggest that a tool needs to be called and langchain runs it


''

{'name': 'multiply',
 'args': {'b': 10, 'a': 3},
 'id': 'call_851656',
 'type': 'tool_call'}

### Tool Execution

In [34]:
multiply.invoke(result.tool_calls[0])   # send tool call result of the previous llm

ToolMessage(content='30', name='multiply', tool_call_id='call_851656')

### Better formatting with Messages

In [51]:
query = HumanMessage('can you multiply 30 with 10')
messages = [query]
messages

[HumanMessage(content='can you multiply 30 with 10', additional_kwargs={}, response_metadata={})]

In [52]:
result = llm_with_tools.invoke(messages)
messages.append(result)
messages

[HumanMessage(content='can you multiply 30 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 30, "b": 10}'}, '__gemini_function_call_thought_signatures__': {'call_544454': 'EnEKbwFpFH0T79pPRrpRGtstYOR1NI78j49CzqRwJE+Z+HshQtSgXz7LJo9YxCqTn8lDoRWvlVQKUOGsPK1oScz7f7TxC7XlCl6ZZJjkUmV8lH/MFe6NMNttoNyT3w1IytNEwTOOCPSUBTQGqED8TEDpXQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d32a-d0c0-7833-8d9e-6f210f209adb-0', tool_calls=[{'name': 'multiply', 'args': {'a': 30, 'b': 10}, 'id': 'call_544454', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 71, 'output_tokens': 18, 'total_tokens': 89, 'input_token_details': {'cache_read': 0}})]

In [54]:
tool_result = multiply.invoke(result.tool_calls[0])
messages.append(tool_result)
messages

[HumanMessage(content='can you multiply 30 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 30, "b": 10}'}, '__gemini_function_call_thought_signatures__': {'call_544454': 'EnEKbwFpFH0T79pPRrpRGtstYOR1NI78j49CzqRwJE+Z+HshQtSgXz7LJo9YxCqTn8lDoRWvlVQKUOGsPK1oScz7f7TxC7XlCl6ZZJjkUmV8lH/MFe6NMNttoNyT3w1IytNEwTOOCPSUBTQGqED8TEDpXQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d32a-d0c0-7833-8d9e-6f210f209adb-0', tool_calls=[{'name': 'multiply', 'args': {'a': 30, 'b': 10}, 'id': 'call_544454', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 71, 'output_tokens': 18, 'total_tokens': 89, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='300', name='multiply', tool_call_id='call_544454'),
 ToolMessage(content='300', name='multiply', 

In [56]:
llm_with_tools.invoke(messages).text

'30 multiplied by 10 is 300.'